In [19]:
#son hal 

In [26]:
import pandas as pd
from scipy.sparse import csr_matrix
from sklearn.metrics.pairwise import cosine_similarity

df = pd.read_parquet("../data/processed/fact_orders.parquet")

In [27]:
category_col = "product_category_name_english" if "product_category_name_english" in df.columns else "product_category_name"

In [28]:
df_cat = df[["order_id", category_col]].dropna().drop_duplicates().copy()

df_cat[category_col] = (
    df_cat[category_col]
    .astype(str)
    .str.strip()
    .str.replace("_", " ", regex=False)
    .str.title()
)

In [29]:
order_counts = df_cat.groupby("order_id", observed=True)[category_col].nunique()
multi_orders = order_counts[order_counts > 1].index
df_cat = df_cat[df_cat["order_id"].isin(multi_orders)].copy()

In [30]:
df_cat["order_id"] = df_cat["order_id"].astype("category")
df_cat[category_col] = df_cat[category_col].astype("category")

In [31]:
df_cat["order_id"] = df_cat["order_id"].cat.remove_unused_categories()
df_cat[category_col] = df_cat[category_col].cat.remove_unused_categories()

In [32]:
row = df_cat[category_col].cat.codes
col = df_cat["order_id"].cat.codes

In [33]:
n_categories = len(df_cat[category_col].cat.categories)
n_orders = len(df_cat["order_id"].cat.categories)

In [34]:
category_order_matrix = csr_matrix(
    ([1] * len(df_cat), (row, col)),
    shape=(n_categories, n_orders)
)

In [35]:

similarity = cosine_similarity(category_order_matrix, dense_output=False)
category_names = df_cat[category_col].cat.categories

In [42]:
def recommend_categories(category_name, top_n=5):
    if category_name not in category_names:
        return pd.DataFrame(columns=["recommended_category", "similarity_score"])

    idx = category_names.get_loc(category_name)
    sim_scores = similarity[idx].toarray().flatten()
    similar_idx = sim_scores.argsort()[::-1]

    recommendations = pd.DataFrame({
        "recommended_category": category_names[similar_idx],
        "similarity_score": sim_scores[similar_idx]
    })

    recommendations = recommendations[
        recommendations["recommended_category"] != category_name
    ]

    recommendations = recommendations[
        recommendations["similarity_score"] > 0
    ]

    recommendations = recommendations.drop_duplicates("recommended_category")

    return recommendations.head(top_n).reset_index(drop=True)

In [52]:
def recommendation_text(category_name, top_n=5):
    recs = recommend_categories(category_name, top_n=top_n)

    return {
        "selected_category": category_name,
        "title": f"Customers who bought {category_name} also bought:",
        "items": recs["recommended_category"].tolist()
    }

In [50]:
result = recommendation_text("Auto", top_n=5)
print(result)

{'selected_category': 'Auto', 'title': 'Customers who bought Auto also bought', 'items': ['Christmas Supplies', 'Construction Tools Construction', 'Computers Accessories', 'Home Comfort 2', 'Telephony']}


# NOTE:
# Bu dosya recommendation dashboardu için kullanılıyor.
# Seçilen kategoriye göre benzer kategoriler öneriliyor.
# Cosine similarity kullanılarak kategori benzerliği hesaplanıyor.
#
# Dashboardu geliştirmek isteyen kişi:
# - kartlara görsel ekleyebilir
# - grafik ekleyebilir
# - ürün bazlı öneri sistemi ekleyebilir